# Step 1 — Clean repository metadata

> Normalize fields, validate values, and deduplicate the raw repository list from Cardoen et al. (2024) so each row corresponds to a unique repository.

**Paper section:** Step 1 — Repository selection and workflow extraction (offline preparation).

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/a-dataset-of-github-actions-workflow-histories-repositories.csv` | CSV (~5.2 MB) | External: Cardoen et al. (2024) workflow-histories dataset |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/1_cleaned_repositories.csv` | CSV (one row per repository) | Step 3 (merge with workflows) |
| `data/step1_report.json` | JSON (run metadata + SHA-256) | Step 3 (integrity check) |




In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 1 — starting: clean repository metadata...", flush=True)
"""Step 1: clean and deduplicate repositories dataset.

Reads the raw repositories CSV, applies hygiene checks, and outputs a clean
version that will be used for the merge step.
"""

import argparse
import json
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

import polars as pl

NULL_TOKENS = {"", "null", "none", "nan", "na", "n/a", "undefined", "-"}
_MSG_NO_HEADER = "Missing CSV header"
NUMERIC_COLUMNS = [
    "commits",
    "branches",
    "releases",
    "contributors",
    "stars",
    "issues",
    "pullrequests",
    "codelines",
    "size",
]
DATE_COLUMNS = ["created", "updated"]
REQUIRED_COLUMNS = ["name", "branch", "language", "created", "updated"]
DATE_FORMATS = [
    "%Y-%m-%d",
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%dT%H:%M:%S",
    "%Y-%m-%dT%H:%M:%SZ",
]

@dataclass
class CleaningStats:
    total_rows: int = 0
    rows_written: int = 0
    removed_exact_duplicates: int = 0
    removed_repo_duplicates: int = 0
    removed_missing_required: int = 0
    removed_invalid_numeric: int = 0
    removed_invalid_date: int = 0
    removed_inverted_dates: int = 0
    trimmed_values: int = 0
    normalized_nulls: int = 0
    numeric_normalized: int = 0
    date_normalized: int = 0

def normalize_text(value: str | None) -> str:
    if value is None:
        return ""
    trimmed = value.strip()
    if trimmed.lower() in NULL_TOKENS:
        return ""
    return trimmed

def normalize_number(value: str) -> int | None:
    if value == "" or value is None:
        return None
    try:
        float_val = float(value)
    except (ValueError, TypeError):
        return None
    if float_val < 0:
        return None
    if float_val != int(float_val):
        return None
    return int(float_val)

def normalize_date(value: str) -> str | None:
    if value == "" or value is None:
        return None
    for fmt in DATE_FORMATS:
        try:
            parsed = datetime.strptime(value, fmt)
            return parsed.strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

def clean_dataset(
    input_path: Path, output_path: Path, *, deduplicate_by_repo: bool = True
) -> CleaningStats:
    stats = CleaningStats()

    from dataset_utils.integrity import hash_source_file
    input_sha = hash_source_file(input_path, "step 1 input")

    print("  reading input CSV (may take a moment for large files)...", flush=True)
    df = pl.read_csv(input_path, encoding="utf-8-sig")

    fieldnames = df.columns
    stats.total_rows = len(df)

    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df = df.with_columns(
                pl.when(
                    pl.col(col).is_null()
                    | (pl.col(col).cast(pl.Utf8, strict=False) == "")
                )
                .then(pl.lit(None).cast(pl.Float64))
                .otherwise(pl.col(col).cast(pl.Float64, strict=False))
                .alias(col)
            )

    text_cols = [c for c in df.columns if c not in NUMERIC_COLUMNS + DATE_COLUMNS]
    for col in text_cols:
        df = df.with_columns(
            pl.col(col).map_elements(normalize_text, return_dtype=pl.String).alias(col)
        )

    missing_mask = pl.lit(False)
    for col in REQUIRED_COLUMNS:
        if col in df.columns:
            missing_mask = missing_mask | (pl.col(col).is_null() | (pl.col(col) == ""))
    stats.removed_missing_required = df.select(missing_mask.sum()).item()

    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col).map_elements(normalize_number, return_dtype=pl.Int64).alias(col)
            )
            null_count = df.filter(pl.col(col).is_null()).height
            stats.removed_invalid_numeric += null_count

    for col in DATE_COLUMNS:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col).map_elements(normalize_date, return_dtype=pl.String).alias(col)
            )
            null_count = df.filter(pl.col(col).is_null()).height
            stats.removed_invalid_date += null_count

    if "created" in df.columns and "updated" in df.columns:
        date_check = df.filter(
            pl.col("updated").str.to_date("%Y-%m-%d") < pl.col("created").str.to_date("%Y-%m-%d")
        )
        stats.removed_inverted_dates = date_check.height
        df = df.filter(
            pl.col("updated").str.to_date("%Y-%m-%d") >= pl.col("created").str.to_date("%Y-%m-%d")
        )

    df = df.filter(~missing_mask)

    row_hash_expr = pl.struct([pl.col(c) for c in fieldnames]).hash()
    df = df.with_columns(row_hash_expr.alias("_row_hash"))
    unique_hashes = df["_row_hash"].unique()
    stats.removed_exact_duplicates = stats.total_rows - len(unique_hashes)
    df = df.unique(subset=["_row_hash"])
    df = df.drop("_row_hash")

    if deduplicate_by_repo and "name" in df.columns:
        df = df.sort(["name", "updated", "commits"], descending=[False, True, False])
        df = df.unique(subset=["name"], keep="first")
        stats.removed_repo_duplicates = stats.total_rows - stats.removed_exact_duplicates - len(df)

    stats.rows_written = len(df)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    print("  writing output...", flush=True)
    df.write_csv(output_path, include_header=True)

    from dataset_utils.hashing import compute_file_hash
    report = {
        **asdict(stats),
        "input_sha256": input_sha,
        "output_file": str(output_path),
        "output_sha256": compute_file_hash(output_path),
    }
    report_path = output_path.parent / "step1_report.json"
    with report_path.open("w") as f:
        json.dump(report, f, indent=2)

    return stats

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Clean and deduplicate repositories dataset.")
    parser.add_argument(
        "--input",
        type=Path,
        default=Path("data/a-dataset-of-github-actions-workflow-histories-repositories.csv"),
        help="Input repositories CSV file path.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/1_cleaned_repositories.csv"),
        help="Output cleaned repositories CSV file path.",
    )
    parser.add_argument(
        "--no-deduplicate-repo",
        action="store_true",
        default=False,
        help="Disable repository-level deduplication.",
    )
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    stats = clean_dataset(args.input, args.output, deduplicate_by_repo=not args.no_deduplicate_repo)

    print("Repository cleaning finished.")
    print(f"Input: {args.input}")
    print(f"Output: {args.output}")
    print("-" * 40)
    print(f"Total rows read: {stats.total_rows}")
    print(f"Rows written: {stats.rows_written}")
    print("-" * 40)
    print(f"Removed exact duplicate rows: {stats.removed_exact_duplicates}")
    print(f"Removed repository duplicate rows: {stats.removed_repo_duplicates}")
    print(f"Removed missing required fields: {stats.removed_missing_required}")
    print(f"Nulled invalid numeric values (rows kept): {stats.removed_invalid_numeric}")
    print(f"Nulled invalid dates (rows kept): {stats.removed_invalid_date}")
    print(f"Removed inverted dates: {stats.removed_inverted_dates}")

main()


Step 1 — starting: clean repository metadata...
  reading input CSV (may take a moment for large files)...
  writing output...
Repository cleaning finished.
Input: data/a-dataset-of-github-actions-workflow-histories-repositories.csv
Output: data/1_cleaned_repositories.csv
----------------------------------------
Total rows read: 49510
Rows written: 49510
----------------------------------------
Removed exact duplicate rows: 0
Removed repository duplicate rows: 0
Removed missing required fields: 0
Nulled invalid numeric values (rows kept): 0
Nulled invalid dates (rows kept): 0
Removed inverted dates: 0


---

**Next step:** [`02_deduplicate_workflows.ipynb`](./02_deduplicate_workflows.ipynb)
